# 03 — Train Classifier Model
Fine-tunes RoBERTa to classify each sentence by **event type**: `none`, `expiration`, `effective`, or `renewal`.

- `deadline_type` (explicit vs computable) is derived downstream from NER output — not from this classifier.
- **Requires:** `01_build_dataset.ipynb` to have been run first (creates `../data/deadline_sentences/`).

## Cell 1 — Imports

In [ ]:
import os, time, platform, random
import torch
import torch.nn as nn
import numpy as np
from collections import Counter
from datasets import load_from_disk
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score as skf1, accuracy_score, classification_report
import mlflow, mlflow.pytorch

print('PyTorch    :', torch.__version__)
print('CUDA avail :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU        :', torch.cuda.get_device_name(0))
print('Imports OK')

## Cell 2 — MLflow & Environment Config

In [ ]:
os.environ['AWS_ACCESS_KEY_ID']      = 'datanauts-key'
os.environ['AWS_SECRET_ACCESS_KEY']  = 'datanauts-secret'
os.environ['MLFLOW_S3_ENDPOINT_URL'] = 'http://129.114.27.190:9000'
os.environ['GIT_PYTHON_REFRESH']     = 'quiet'

MLFLOW_URI = 'http://129.114.27.190:8000'
EXPERIMENT = 'deadline-detection-classifier'
DATA_PATH  = '../data/deadline_sentences'
OUTPUT_DIR = '/tmp/deadline-clf'

mlflow.set_tracking_uri(MLFLOW_URI)
print('MLflow URI :', MLFLOW_URI)
print('Data path  :', DATA_PATH)

## Cell 3 — Label Schema & Model Configs

In [ ]:
CLF_LABEL2ID = {'none': 0, 'expiration': 1, 'effective': 2, 'renewal': 3}
CLF_ID2LABEL = {0: 'none', 1: 'expiration', 2: 'effective', 3: 'renewal'}
NUM_LABELS   = 4

CONFIGS = {
    'baseline':       {'base_model': 'roberta-base', 'epochs': 0, 'learning_rate': 0,    'batch_size': 16, 'max_seq_length': 256, 'fp16': False, 'none_ratio': 5},
    'roberta_clf_v1': {'base_model': 'roberta-base', 'epochs': 3, 'learning_rate': 2e-5, 'batch_size': 16, 'max_seq_length': 256, 'fp16': True,  'none_ratio': 5},
    'roberta_clf_v2': {'base_model': 'roberta-base', 'epochs': 3, 'learning_rate': 5e-5, 'batch_size': 16, 'max_seq_length': 256, 'fp16': True,  'none_ratio': 5},
    'roberta_clf_v3': {'base_model': 'roberta-base', 'epochs': 5, 'learning_rate': 2e-5, 'batch_size': 16, 'max_seq_length': 256, 'fp16': True,  'none_ratio': 3},
    'roberta_clf_v4': {'base_model': 'roberta-base', 'epochs': 4, 'learning_rate': 2e-5, 'batch_size': 16, 'max_seq_length': 256, 'fp16': True,  'none_ratio': 8},
}

# ── CHANGE THIS to switch variants ──────────────────────────────────────────
MODEL_VARIANT = 'roberta_clf_v1'
# ────────────────────────────────────────────────────────────────────────────

cfg = CONFIGS[MODEL_VARIANT]
print(f'Selected variant : {MODEL_VARIANT}')
print(f'Config           : {cfg}')

## Cell 4 — Seed & Load Data

In [ ]:
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seeds(42)

dd = load_from_disk(DATA_PATH)

print('=== Raw split sizes ===')
for name in ['train', 'val', 'test']:
    c = Counter(dd[name]['classifier_label'])
    print(f'  [{name}]  {len(dd[name]):>6} sentences | '
          f'none={c[0]:>5}  expiration={c[1]:>4}  effective={c[2]:>4}  renewal={c[3]:>4}')

## Cell 5 — Downsample None Class (fixes class imbalance)

In [ ]:
def downsample_none(ds, none_ratio, seed=42):
    labels   = [int(x) for x in ds['classifier_label']]
    pos_idx  = [i for i, l in enumerate(labels) if l != 0]
    none_idx = [i for i, l in enumerate(labels) if l == 0]
    max_none = min(len(none_idx), none_ratio * len(pos_idx))
    random.seed(seed)
    keep = sorted(random.sample(none_idx, max_none) + pos_idx)
    return ds.select(keep)

train_ds = downsample_none(dd['train'], cfg['none_ratio'])
val_ds   = dd['val']
test_ds  = dd['test']

print(f'none_ratio = {cfg["none_ratio"]}  (keep at most {cfg["none_ratio"]}x positives as none)')
print('=== After downsampling ===')
for name, ds in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    c = Counter([int(x) for x in ds['classifier_label']])
    print(f'  [{name}]  {len(ds):>6} sentences | '
          f'none={c[0]:>5}  expiration={c[1]:>4}  effective={c[2]:>4}  renewal={c[3]:>4}')

## Cell 6 — Compute Class Weights

In [ ]:
def compute_class_weights(ds):
    labels  = [int(x) for x in ds['classifier_label']]
    counts  = Counter(labels)
    total   = sum(counts.values())
    weights = [total / (NUM_LABELS * max(counts.get(i, 1), 1)) for i in range(NUM_LABELS)]
    return torch.tensor(weights, dtype=torch.float)

class_weights = compute_class_weights(train_ds)
print('Class weights:')
for i, (label, w) in enumerate(zip(CLF_ID2LABEL.values(), class_weights)):
    print(f'  {label:<12}: {w:.4f}')
print('\nHigher weight = errors on this class cost more in the loss function')

## Cell 7 — Load Tokenizer & Tokenize Dataset

In [ ]:
if MODEL_VARIANT == 'baseline':
    print('Baseline mode — skipping tokenization')
    tokenizer = None
    tok_train = tok_val = tok_test = None
else:
    tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'])
    print(f'Loaded tokenizer: {cfg["base_model"]}')

    def tokenize(examples):
        return tokenizer(examples['sentence'], truncation=True,
                         max_length=cfg['max_seq_length'], padding=False)

    remove_cols = [c for c in train_ds.column_names if c != 'classifier_label']

    tok_train = train_ds.map(tokenize, batched=True, remove_columns=remove_cols).rename_column('classifier_label', 'label')
    tok_val   = val_ds.map(tokenize,   batched=True, remove_columns=remove_cols).rename_column('classifier_label', 'label')
    tok_test  = test_ds.map(tokenize,  batched=True, remove_columns=remove_cols).rename_column('classifier_label', 'label')

    tok_train.set_format('torch')
    tok_val.set_format('torch')
    tok_test.set_format('torch')

    print(f'Tokenized — train: {len(tok_train)} | val: {len(tok_val)} | test: {len(tok_test)}')

    # Show a sample
    ex = tok_train[0]
    decoded = tokenizer.decode(ex['input_ids'], skip_special_tokens=True)
    print(f'\nSample sentence (label={CLF_ID2LABEL[int(ex["label"])]}): {decoded[:120]}')

## Cell 8 — Majority Class Baseline

In [ ]:
labels_test    = [int(x) for x in test_ds['classifier_label']]
majority_class = Counter(labels_test).most_common(1)[0][0]
preds_majority = [majority_class] * len(labels_test)

baseline_f1  = skf1(labels_test, preds_majority, average='macro', zero_division=0)
baseline_acc = accuracy_score(labels_test, preds_majority)

print(f'Majority class     : {CLF_ID2LABEL[majority_class]}')
print(f'Baseline F1 (macro): {baseline_f1:.4f}')
print(f'Baseline Accuracy  : {baseline_acc:.4f}')
print('\nLabel distribution in test set:')
for k, v in sorted(Counter(labels_test).items()):
    print(f'  {CLF_ID2LABEL[k]:<12}: {v:>5}  ({100*v/len(labels_test):.1f}%)')

## Cell 9 — WeightedTrainer Definition

In [ ]:
class WeightedTrainer(Trainer):
    """Trainer with weighted CrossEntropy loss to handle none-class imbalance."""
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        w       = self.class_weights.to(logits.device)
        loss    = nn.CrossEntropyLoss(weight=w)(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_clf_metrics(p):
    preds  = np.argmax(p.predictions, axis=1)
    labels = p.label_ids
    return {
        'accuracy':    float(accuracy_score(labels, preds)),
        'f1':          float(skf1(labels, preds, average='macro',    zero_division=0)),
        'f1_weighted': float(skf1(labels, preds, average='weighted', zero_division=0)),
    }

print('WeightedTrainer defined')

## Cell 10 — Train Model
> Skip this cell if `MODEL_VARIANT = 'baseline'`

In [ ]:
if MODEL_VARIANT == 'baseline':
    print('Baseline variant — no model training needed.')
    model = None; train_result = None; train_time = 0
else:
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg['base_model'], num_labels=NUM_LABELS,
        id2label=CLF_ID2LABEL, label2id=CLF_LABEL2ID,
    )
    t_args = TrainingArguments(
        output_dir=f'{OUTPUT_DIR}-{MODEL_VARIANT}',
        num_train_epochs=cfg['epochs'],
        per_device_train_batch_size=cfg['batch_size'],
        per_device_eval_batch_size=cfg['batch_size'],
        learning_rate=cfg['learning_rate'],
        weight_decay=0.01,
        warmup_steps=50,
        fp16=cfg['fp16'] and torch.cuda.is_available(),
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        logging_steps=20,
        report_to='none',
    )
    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model, args=t_args,
        train_dataset=tok_train, eval_dataset=tok_val,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_clf_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    print(f'Training {MODEL_VARIANT}  |  lr={cfg["learning_rate"]}  |  epochs={cfg["epochs"]}  |  fp16={cfg["fp16"] and torch.cuda.is_available()}')
    t0 = time.time()
    train_result = trainer.train()
    train_time = time.time() - t0
    print(f'\nTraining complete in {train_time:.1f}s')
    print(f'Final train loss: {train_result.training_loss:.4f}')

## Cell 11 — Evaluate on Test Set

In [ ]:
if MODEL_VARIANT == 'baseline':
    test_result = {'eval_f1': baseline_f1, 'eval_accuracy': baseline_acc, 'eval_loss': 0}
    report = ''
    print(f'Baseline  F1: {baseline_f1:.4f} | Accuracy: {baseline_acc:.4f}')
else:
    print('Evaluating on test set...')
    preds_out = trainer.predict(tok_test)
    preds     = np.argmax(preds_out.predictions, axis=1)
    labels    = preds_out.label_ids
    report    = classification_report(
        labels, preds,
        target_names=[CLF_ID2LABEL[i] for i in range(NUM_LABELS)],
        zero_division=0,
    )
    test_result = {
        'eval_f1':       float(skf1(labels, preds, average='macro',    zero_division=0)),
        'eval_accuracy': float(accuracy_score(labels, preds)),
        'eval_loss':     preds_out.metrics.get('test_loss', 0),
    }
    print(report)
    print(f'Test F1 (macro) : {test_result["eval_f1"]:.4f}')
    print(f'Test Accuracy   : {test_result["eval_accuracy"]:.4f}')
    print(f'Baseline F1     : {baseline_f1:.4f}  (delta: {test_result["eval_f1"]-baseline_f1:+.4f})')

## Cell 12 — Log to MLflow

In [ ]:
mlflow.set_experiment(EXPERIMENT)
run_name = MODEL_VARIANT if MODEL_VARIANT != 'baseline' else 'clf_baseline_majority'

with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_params({
        'model_name':     run_name,
        'base_model':     cfg['base_model'],
        'task':           'sequence_classification',
        'num_labels':     NUM_LABELS,
        'label_schema':   'none|expiration|effective|renewal',
        'epochs':         cfg['epochs'],
        'batch_size':     cfg['batch_size'],
        'learning_rate':  cfg['learning_rate'],
        'max_seq_length': cfg['max_seq_length'],
        'fp16':           cfg['fp16'],
        'none_ratio':     cfg['none_ratio'],
        'train_size':     len(train_ds),
        'val_size':       len(val_ds),
        'test_size':      len(test_ds),
        'gpu':            torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'pytorch':        torch.__version__,
        'platform':       platform.platform(),
    })
    mlflow.log_metrics({
        'baseline_f1':          baseline_f1,
        'total_train_time_sec': train_time if MODEL_VARIANT != 'baseline' else 0,
        'train_loss':           train_result.training_loss if train_result else 0,
        'test_f1':              test_result['eval_f1'],
        'test_accuracy':        test_result['eval_accuracy'],
        'test_loss':            test_result['eval_loss'],
    })
    if report:
        mlflow.log_text(report, 'clf_classification_report.txt')

    run_url = f'{MLFLOW_URI}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}'
    print(f'Logged run : {run_name}')
    print(f'MLflow URL : {run_url}')